## NVDA Time Series – Main Notebook

This notebook replaces the previous `main.py` script as the primary interactive entry point for the NVDA projects.

Use it to:
- Download and load the NVDA dataset.
- Run preprocessing and basic data description.
- Inspect the DataFrame with commands like `df.head()`.
- Later: perform transformations to returns and add visualizations (after we verify the time-series structure).


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = yf.download(
    tickers="NVDA",
    start="2021-01-01",
    end="2026-03-05",
    auto_adjust=True,
)

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.index = pd.to_datetime(df.index)
print(df.shape)
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDate range:", df.index.min(), "→", df.index.max())
df.describe()

In [ ]:
# Drop duplicated index rows if any
df = df[~df.index.duplicated(keep="first")]

# Drop rows where Close is missing (the target column)
df = df.dropna(subset=["Close"])

# Forward-fill any remaining gaps (e.g. sparse Volume on holidays)
df = df.ffill()

print("Shape after cleaning:", df.shape)
print("Missing values:\n", df.isnull().sum())

In [ ]:
df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

In [ ]:
df["Lag_1"] = df["Log_Return"].shift(1)
df["Lag_2"] = df["Log_Return"].shift(2)
df["Lag_5"] = df["Log_Return"].shift(5)

In [ ]:
df["Volatility_10"] = df["Log_Return"].rolling(window=10).std()
df["Volatility_20"] = df["Log_Return"].rolling(window=20).std()

In [ ]:
rolling_mean = df["Close"].rolling(window=20).mean()
rolling_std  = df["Close"].rolling(window=20).std()
bb_upper     = rolling_mean + (2 * rolling_std)
bb_lower     = rolling_mean - (2 * rolling_std)
bb_width     = bb_upper - bb_lower

df["BB_Pct"] = (df["Close"] - bb_lower) / bb_width

In [ ]:
df["Volume_Change"] = df["Volume"].pct_change()

In [ ]:
sp500 = yf.download("^GSPC", start="2021-01-01", end="2026-03-05", auto_adjust=True)["Close"].squeeze()
vix   = yf.download("^VIX",  start="2021-01-01", end="2026-03-05", auto_adjust=True)["Close"].squeeze()

df["SP500_Return"] = np.log(sp500 / sp500.shift(1))
df["VIX"] = vix

In [ ]:
df["Target"] = df["Log_Return"].shift(-1)

In [ ]:
df = df.dropna()

print("Final shape:", df.shape)
print("Missing values:\n", df.isnull().sum())